# RiskQuant: Valuation

In [ ]:
%load_ext IPython.extensions.autoreload

In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Data Validation extension is not supported",
    module="openpyxl"
)

## General Imports

In [ ]:
from typing import Final
import numpy as np

## RiskQuant Imports

In [ ]:
from riskquant.utils.printer import (
    print_as_list_str, 
    print_df, 
    print_dict, 
    print_success_message, 
    print_wrapped_text_box,
    )

## Spot Pricing

In [ ]:
# ---> IMPORTS <--- #
from typing import Final
import numpy as np

from riskquant.portfolio.blocks.spot_block import SpotBlock
from riskquant.market_data.state import MarketDataMatrix
from riskquant.portfolio.compiled import CompiledPortfolio
from riskquant.valuation.engine import run_valuation

# --> PARAMS <--- #
N_TRADES: Final[int] = 10
S_SCENARIOS: Final[int] = 5
M_PRICE_RFS: Final[int] = 20
F_FX_SPOT_RFS: Final[int] = 5
RNG_SEED: Final[int] = 69
rng = np.random.default_rng(RNG_SEED)

trade_ids = rng.choice(np.arange(100 * N_TRADES, dtype=np.int64), size=N_TRADES, replace=False)

# ---> 1. THE "EDGE" CONTEXT (Human Readable) <--- #
# This represents what the system knows BEFORE compiling into DOD arrays
edge_price_tickers = {i: f"TICKER_{i}" for i in range(M_PRICE_RFS)}
edge_fx_tickers = {i: f"CCY_{i}USD" for i in range(F_FX_SPOT_RFS)}

# ---> BLOCK <--- #
spot_block = SpotBlock(
    trade_uids=trade_ids,
    qty=rng.integers(low=-100, high=+100, size=N_TRADES).astype(np.float64),
    price_uix=rng.choice(np.arange(M_PRICE_RFS), size=N_TRADES, replace=True).astype(np.int32),    
    fx_uix=rng.choice(np.arange(F_FX_SPOT_RFS), size=N_TRADES, replace=True).astype(np.int32),
    fx_invert=rng.choice([0, 1], size=N_TRADES, replace=True).astype(np.int8),
)

# ---> MARKET DATA <--- #
# Instead of 1.0, we make each row distinct. 
# Ticker 0 is ~100, Ticker 1 is ~200, etc.
base_prices = np.arange(1, M_PRICE_RFS + 1) * 100.0
prices_mock = base_prices[:, None] + rng.uniform(-2.0, 2.0, (M_PRICE_RFS, S_SCENARIOS)).round(0)

# CCY 0 is ~1.1, CCY 1 is ~1.2, etc.
base_fx = np.arange(1, F_FX_SPOT_RFS + 1) * 1.1
fx_mock = base_fx[:, None] + rng.uniform(-0.02, 0.02, (F_FX_SPOT_RFS, S_SCENARIOS)).round(2)

md = MarketDataMatrix(
    num_scenarios=S_SCENARIOS,
    fx_spots=fx_mock,
    prices=prices_mock,
)

# ---> PORTFOLIO <--- # 
portfolio = CompiledPortfolio(spot=spot_block)
portfolio.summary()

# ---> VALUATION <--- #
pvs = run_valuation(portfolio=portfolio, market_data_mtx=md)

print("\n" + "="*80)
print("PV OUTPUT MATRIX (10 Trades x 5 Scenarios)")
print("="*80)
print(pvs.round(2))

# ---> THE AUDIT TRAIL <--- #
for trade_idx in range(N_TRADES):
    print("\n\n" + "="*80)
    print(f"AUDIT TRAIL: ISOLATING TRADE INDEX {trade_idx}")
    print("="*80)

    # Extract trade parameters
    t_qty = spot_block.qty[trade_idx]
    p_uix = spot_block.price_uix[trade_idx]
    f_uix = spot_block.fx_uix[trade_idx]
    f_inv = spot_block.fx_invert[trade_idx]

    print(f"Trade Data:")
    print(f"  Quantity:       {t_qty}")
    print(f"  Price UIX:      {p_uix} (Points to {edge_price_tickers[p_uix]})")
    print(f"  FX UIX:         {f_uix} (Points to {edge_fx_tickers[f_uix]})")
    print(f"  FX Invert Flag: {bool(f_inv)}")

    print(f"\nEngine Memory Lookups for Trade {trade_idx}:")
    print(f"  md.prices[{p_uix}, :]   -> {md.prices[p_uix, :].round(2)}")
    print(f"  md.fx_spots[{f_uix}, :] -> {md.fx_spots[f_uix, :].round(2)}")

    # Perform the pure-python manual calculation
    manual_prices = md.prices[p_uix, :]
    manual_fx = md.fx_spots[f_uix, :]
    if f_inv:
        manual_fx = 1.0 / manual_fx

    manual_pv = t_qty * manual_prices * manual_fx

    print(f"\nVerification:")
    print(f"  Engine Calculated PVs: {pvs[trade_idx, :].round(2)}")
    print(f"  Hand-Calculated PVs:   {manual_pv.round(2)}")
    print(f"  Match?                 {np.allclose(pvs[trade_idx, :], manual_pv)}")

## Forward Node Pricing

In [ ]:
# ---> IMPORTS <--- #
from typing import Final
import numpy as np

from riskquant.portfolio.blocks.forward_block import ForwardNodeBlock
from riskquant.market_data.state import MarketDataMatrix
from riskquant.portfolio.compiled import CompiledPortfolio
from riskquant.valuation.engine import run_valuation

# --> PARAMS <--- #
M_FWD_RFS: Final[int] = 10
C_CURVES: Final[int] = 3
D_DATES: Final[int] = 4

trade_ids = np.arange(1000, 1000 + N_TRADES, dtype=np.int64)

# ---> 1. THE "EDGE" CONTEXT <--- #
edge_fwd_tickers = {i: f"FWD_NODE_{i}" for i in range(M_FWD_RFS)}
edge_fx_tickers = {i: f"CCY_{i}USD" for i in range(F_FX_SPOT_RFS)}
edge_curves = {i: f"YIELD_CURVE_{i}" for i in range(C_CURVES)}

# ---> 2. BLOCK CONSTRUCTION <--- #
# Randomize everything first...
qty = rng.uniform(-100, 100, N_TRADES).round(0)
strike = rng.uniform(90, 110, N_TRADES).round(0)
price_uix = rng.integers(0, M_FWD_RFS, N_TRADES, dtype=np.int32)
fx_u_uix = rng.integers(0, F_FX_SPOT_RFS, N_TRADES, dtype=np.int32)
fx_u_inv = rng.integers(0, 2, N_TRADES, dtype=np.int8)
df_u_uix = rng.integers(0, C_CURVES, N_TRADES, dtype=np.int32)
fx_k_uix = rng.integers(0, F_FX_SPOT_RFS, N_TRADES, dtype=np.int32)
fx_k_inv = rng.integers(0, 2, N_TRADES, dtype=np.int8)
df_k_uix = rng.integers(0, C_CURVES, N_TRADES, dtype=np.int32)
stl_uix = rng.integers(0, D_DATES, N_TRADES, dtype=np.int32)

# FORCE TRADE 0 TO BE A COMPLEX QUANTO WITH DISCOUNTING
qty[0] = 50.0
strike[0] = 105.0
price_uix[0] = 2  # FWD_NODE_2
fx_u_uix[0] = 1   # CCY_1USD (Underlying FX)
fx_u_inv[0] = 0   # Multiply
df_u_uix[0] = 0   # YIELD_CURVE_0
fx_k_uix[0] = 3   # CCY_3USD (Strike FX)
fx_k_inv[0] = 1   # Invert (Divide)
df_k_uix[0] = 2   # YIELD_CURVE_2
stl_uix[0] = 1    # Settle Date Index 1

fwd_block = ForwardNodeBlock(
    trade_uids=trade_ids, qty=qty, strike=strike, price_uix=price_uix,
    fx_under_to_settle_uix=fx_u_uix, fx_under_to_settle_invert=fx_u_inv,
    under_discount_curve_uix=df_u_uix, fx_strike_to_settle_uix=fx_k_uix,
    fx_strike_to_settle_invert=fx_k_inv, strike_discount_curve_uix=df_k_uix,
    settlement_date_uix=stl_uix
)

# ---> 3. MARKET DATA <--- #

# FX Spots: Each row is a different FX pair, each column is a scenario.
fx_spots = np.arange(1, F_FX_SPOT_RFS + 1)[:, None] + rng.uniform(-0.05, 0.05, (F_FX_SPOT_RFS, S_SCENARIOS)).round(2)

# Forward Nodes: Each row is a different forward node, each column is a scenario.
forward_nodes = (np.arange(1, M_FWD_RFS + 1)[:, None] * 100.0 + rng.uniform(-2, 2, (M_FWD_RFS, S_SCENARIOS))).round(2)

# MD
md = MarketDataMatrix(
    num_scenarios=S_SCENARIOS,
    fx_spots=fx_spots,
    prices=np.empty((0, 0)), # Unused
    forward_nodes=forward_nodes,
    discount_factor_grid=rng.uniform(0.95, 0.99, (C_CURVES, D_DATES, S_SCENARIOS)).round(2),
)

# ---> 4.1 PORTFOLIO <--- #
portfolio = CompiledPortfolio(forward_node=fwd_block)
portfolio.summary()

# ---> 4.2 RUN VALUATION <--- #
pvs = run_valuation(portfolio=portfolio, market_data_mtx=md)

print("\n" + "="*80)
print("PV OUTPUT MATRIX (10 Trades x 5 Scenarios)")
print("="*80)
print(pvs.round(2))


# ---> 5. THE AUDIT TRAIL <--- #
for trade_idx in range(N_TRADES):
    print("\n\n" + "="*80)
    print(f"AUDIT TRAIL: ISOLATING TRADE INDEX {trade_idx}")
    print("="*80)

    # Extract base parameters
    t_qty = fwd_block.qty[trade_idx]
    t_strike = fwd_block.strike[trade_idx]
    settlement_date_uix = fwd_block.settlement_date_uix[trade_idx]
    fx_u2s = fwd_block.fx_under_to_settle_uix[trade_idx]
    fx_k2s_uix = fwd_block.fx_strike_to_settle_uix[trade_idx]
    fx_u2s_inv = fwd_block.fx_under_to_settle_invert[trade_idx]
    fx_k2s_inv = fwd_block.fx_strike_to_settle_invert[trade_idx]
    under_disc_uix = fwd_block.under_discount_curve_uix[trade_idx]
    strike_disc_uix = fwd_block.strike_discount_curve_uix[trade_idx]

    # Node parameters
    p_uix = fwd_block.price_uix[trade_idx]

    # Display Trade Configuration
    print(f"Trade Data:")
    print(f"  Quantity:                  {t_qty:.2f}")
    print(f"  Strike:                    {t_strike:.2f}")
    print(f"  Settlement Date UIX:       {settlement_date_uix}")
    print(f"  U2S FX UIX:                {fx_u2s} (Invert: {bool(fx_u2s_inv)}) -> {edge_fx_tickers[fx_u2s]}")
    print(f"  K2S FX UIX:                {fx_k2s_uix} (Invert: {bool(fx_k2s_inv)}) -> {edge_fx_tickers[fx_k2s_uix]}")
    print(f"  Underlying Disc Curve UIX: {under_disc_uix} -> {edge_curves[under_disc_uix]}")
    print(f"  Strike Disc Curve UIX:     {strike_disc_uix} -> {edge_curves[strike_disc_uix]}")
    print(f"  Forward Node UIX:          {p_uix} -> {edge_fwd_tickers[p_uix]}")

    print(f"\nEngine Memory Lookups for Trade {trade_idx}:")
    print(f"  md.forward_nodes[{p_uix:2.0f}, :]            -> {md.forward_nodes[p_uix, :].round(2)}")
    print(f"  md.fx_spots[{fx_u2s:2.0f}, :] (Underlying)    -> {md.fx_spots[fx_u2s, :].round(4)}")
    print(f"  md.fx_spots[{fx_k2s_uix:2.0f}, :] (Strike)        -> {md.fx_spots[fx_k2s_uix, :].round(4)}")

    # Perform the pure-python manual calculation
    
    # 1. Underlying Leg
    fwd_leg = md.forward_nodes[p_uix, :]
    fx_u = md.fx_spots[fx_u2s, :]
    if fx_u2s_inv: 
        fx_u = 1.0 / fx_u
    df_u = md.discount_factor_grid[under_disc_uix, settlement_date_uix, :]
    
    # 2. Strike Leg
    strike_leg = t_strike
    fx_k = md.fx_spots[fx_k2s_uix, :]
    if fx_k2s_inv: 
        fx_k = 1.0 / fx_k
    df_k = md.discount_factor_grid[strike_disc_uix, settlement_date_uix, :]

    # 3. Final PV
    manual_pv = t_qty * ((fwd_leg * fx_u * df_u) - (strike_leg * fx_k * df_k))

    print(f"\nVerification:")
    print(f"  Engine Calculated PVs: {pvs[trade_idx, :].round(2)}")
    print(f"  Hand-Calculated PVs:   {manual_pv.round(2)}")
    print(f"  Match?                 {np.allclose(pvs[trade_idx, :], manual_pv)}")

In [ ]:
fwd_block.print_block()